In [1]:
# Imports and setup
import sys
from pathlib import Path
# Ensure the repository `src` folder is on sys.path so `from utils...` works when running cells
sys.path.append(str(Path('../../src').resolve()))

import pandas as pd, numpy as np, time, traceback, ast
from pathlib import Path
from joblib import dump
from sklearn.base import clone
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

from utils.models import MODELS
from utils.eval_metrics import evaluate_model

# Paths
BASE_DATA_PATH = Path("../../data/out/dataset_final.csv")
VALIDATION_PATH = Path("../../data/out/dataset_validation_final.csv")
BEST_MODELS = Path("../../data/out/best_models_ml/in/best_models_ml.xlsx")

OUT_DIR = Path("../../data/out/best_models_ml/validation_ml")
MODELS_DIR = OUT_DIR / "models"
PLOTS_DIR = OUT_DIR / "plots"
PRED_DIR  = OUT_DIR / "predictions" 
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)
PRED_DIR.mkdir(exist_ok=True)



TARGET_COLUMN = "PRECIO"
categorical_numeric = ["YEAR","MONTH","DAY","HORA","NIVEL_ENSO","DIA_SEMANA","FESTIVO"]

In [2]:
def prepare_full_data(df, target_column='PRECIO', categorical_numeric=None):
    """
    Prepare X/y and the preprocessor using the entire dataset (no train/test split).
    This is used for training with 100% of the original dataset and then evaluating on an external validation set.
    """
    if categorical_numeric is None:
        categorical_numeric = ["YEAR","MONTH","DAY","HORA","NIVEL_ENSO","DIA_SEMANA","FESTIVO"]

    # Identify numeric features (exclude categorical, target, and datetime column)
    numeric_features = [
        col for col in df.columns
        if col not in categorical_numeric and col != target_column and col != 'FECHA_HORA'
    ]

    # Build ColumnTransformer: scale numeric features, passthrough categorical
    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), numeric_features),
        ('cat', 'passthrough', [c for c in categorical_numeric if c in df.columns])
    ])

    # Separate features and target
    X = df.drop(columns=['FECHA_HORA', target_column], errors='ignore')
    y = df[target_column]

    return preprocessor, X, y

In [3]:
# %% Load datasets
df_train = pd.read_csv(BASE_DATA_PATH)
df_val = pd.read_csv(VALIDATION_PATH)

df_train.head()

,FECHA_HORA,YEAR,MONTH,DAY,HORA,PRECIO,TERMICA,HIDRAULICA,SOLAR,COGENERADOR,...,FUEL_CONS_GAS,FUEL_CONS_GAS_NI,FUEL_CONS_GLP,FUEL_COST_CARBON,FUEL_COST_GAS,FUEL_COST_GAS_NI,FUEL_COST_COMBUSTOLEO,IPC_VAR_MOM_PCT,IPP_VAR_PN_MOM_PCT,IPP_VAR_OI_MOM_PCT
0,2020-01-01 00:00:00,2020,1,1,0,72.02,2109637.93,4418103.51,0.0,28880.74,...,167262.229,0.0,0.0,130.459523,240.075697,0.0,585.085344,0.42,-0.73,-0.02
1,2020-01-01 01:00:00,2020,1,1,1,136.71,1864989.80,4548853.97,0.0,29107.37,...,167262.229,0.0,0.0,130.459523,240.075697,0.0,585.085344,0.42,-0.73,-0.02
2,2020-01-01 02:00:00,2020,1,1,2,127.71,1720883.01,4525898.75,0.0,25939.87,...,167262.229,0.0,0.0,130.459523,240.075697,0.0,585.085344,0.42,-0.73,-0.02
3,2020-01-01 03:00:00,2020,1,1,3,127.71,1649014.61,4458645.56,0.0,25651.57,...,167262.229,0.0,0.0,130.459523,240.075697,0.0,585.085344,0.42,-0.73,-0.02
4,2020-01-01 04:00:00,2020,1,1,4,127.71,1711745.36,4279657.90,0.0,25245.37,...,167262.229,0.0,0.0,130.459523,240.075697,0.0,585.085344,0.42,-0.73,-0.02


In [4]:
df_val.head()

,FECHA_HORA,YEAR,MONTH,DAY,HORA,PRECIO,TERMICA,HIDRAULICA,SOLAR,COGENERADOR,...,FUEL_CONS_GAS,FUEL_CONS_GAS_NI,FUEL_CONS_GLP,FUEL_COST_CARBON,FUEL_COST_GAS,FUEL_COST_GAS_NI,FUEL_COST_COMBUSTOLEO,IPC_VAR_MOM_PCT,IPP_VAR_PN_MOM_PCT,IPP_VAR_OI_MOM_PCT
0,01/07/2025 00:00,2025,7,1,0,104.51,1281388.34,7265679.27,0.0,104102.21,...,53451.40839,144168.5154,0,159.86,512.74,435.23,966.02,0.28,0.39,0.63
1,01/07/2025 01:00,2025,7,1,1,104.51,1240276.15,7001552.24,0.0,107256.21,...,53451.40839,144168.5154,0,159.86,512.74,435.23,966.02,0.28,0.39,0.63
2,01/07/2025 02:00,2025,7,1,2,104.51,1149305.01,6885491.95,0.0,114568.58,...,53451.40839,144168.5154,0,159.86,512.74,435.23,966.02,0.28,0.39,0.63
3,01/07/2025 03:00,2025,7,1,3,104.51,1070921.82,6832299.83,0.0,108933.92,...,53451.40839,144168.5154,0,159.86,512.74,435.23,966.02,0.28,0.39,0.63
4,01/07/2025 04:00,2025,7,1,4,104.51,1025993.51,6962234.35,0.0,108668.60,...,53451.40839,144168.5154,0,159.86,512.74,435.23,966.02,0.28,0.39,0.63


In [5]:
# Separate features and target for training (100% of dataset)
X_full = df_train.drop(columns=[TARGET_COLUMN, "FECHA_HORA"])
y_full = df_train[TARGET_COLUMN]

# Separate features and target for external validation
X_val = df_val.drop(columns=[TARGET_COLUMN, "FECHA_HORA"])
y_val = df_val[TARGET_COLUMN]

# %% Load best models
best_df = pd.read_excel(BEST_MODELS)

rows = []

for _, row in best_df.iterrows():
    variant = row["variant"]
    model_name = row["model"]

    # Parse best_params from Excel
    try:
        best_params = ast.literal_eval(str(row.get("best_params", "{}")))
        if not isinstance(best_params, dict):
            best_params = {}
    except Exception:
        best_params = {}

    try:
        # Clone base estimator and apply best_params
        base_estimator = MODELS[model_name]
        estimator = clone(base_estimator).set_params(**best_params)

        # Build ColumnTransformer for preprocessing
        numeric_features = [
            col for col in X_full.columns
            if col not in categorical_numeric and col != TARGET_COLUMN and col != 'FECHA_HORA'
        ]
        preprocessor = ColumnTransformer([
            ('num', StandardScaler(), numeric_features),
            ('cat', 'passthrough', [c for c in categorical_numeric if c in X_full.columns])
        ])

        # Build pipeline: preprocessor + model
        pipe = Pipeline([("preprocessor", preprocessor), ("model", estimator)])

        # Train with 100% of dataset
        start = time.time()
        pipe.fit(X_full, y_full)
        elapsed = time.time() - start

        # Predict on external validation set
        y_pred = pipe.predict(X_val)
        metrics = evaluate_model(pipe, X_val, y_val)
        metrics.update({
            "variant": variant,
            "model": model_name,
            "train_time_s": elapsed,
            "n_train": len(y_full),
            "n_val": len(y_val),
        })
        rows.append(metrics)

         # Save trained model
        dump(pipe, MODELS_DIR / f"{variant}_{model_name}.joblib")

        # --- Save predictions ---
        df_preds = pd.DataFrame({
            "y_real": y_val,
            "y_pred": y_pred
        })
        df_preds.to_csv(PRED_DIR / f"predictions_{variant}_{model_name}.csv", index=False)


        # Save trained model
        dump(pipe, MODELS_DIR / f"{variant}_{model_name}.joblib")

        # --- Plots ---
        # 1. Predictions vs real values
        plt.figure(figsize=(12,5))
        plt.plot(y_val[:200], label="Real")
        plt.plot(y_pred[:200], label="Predicted")
        plt.legend()
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"pred_{variant}_{model_name}.png")
        plt.close()

        # 2. Scatter with y=x line
        plt.figure(figsize=(5,5))
        plt.scatter(y_val, y_pred, s=6, alpha=0.5)
        lims = [min(plt.xlim()[0], plt.ylim()[0]), max(plt.xlim()[1], plt.ylim()[1])]
        plt.plot(lims, lims, 'r--', linewidth=2)
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"scatter_{variant}_{model_name}.png")
        plt.close()

        # 3. Residuals
        resid = y_val - y_pred
        plt.figure(figsize=(12,4))
        plt.plot(resid[:200], color="tab:red")
        plt.tight_layout()
        plt.savefig(PLOTS_DIR / f"resid_{variant}_{model_name}.png")
        plt.close()

        print(f"OK {variant}-{model_name}: MAE={metrics.get('MAE'):.2f}")

    except Exception as e:
        print(f"ERROR {variant}-{model_name}: {e}")
        traceback.print_exc()

# %% Save consolidated metrics
if rows:
    df_out = pd.DataFrame(rows)
    df_out.to_excel(OUT_DIR / "validation_ml_metrics.xlsx", index=False)


OK v1_original-ElasticNet: MAE=150.42
OK v1_original_lags-Lasso: MAE=158.29
OK v2_with_calendar-ElasticNet: MAE=150.42
OK v2_with_calendar_lags-Lasso: MAE=158.29
OK v3_no_solar-ElasticNet: MAE=149.74
OK v3_no_solar_lags-Lasso: MAE=158.29


c:\Users\Camilo\anaconda3\envs\env_tf\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


OK v4_no_fuel_consumption-MLP Regressor: MAE=70.08
OK v4_no_fuel_consumption_lags-Lasso: MAE=158.29
OK v5_no_fuel_and_cost-XGBoost: MAE=61.62
OK v5_no_fuel_and_cost_lags-Lasso: MAE=158.29
OK v6_no_econ_fuel_cost-MLP Regressor: MAE=54.21
OK v6_no_econ_fuel_cost_lags-Lasso: MAE=158.29
OK v7_only_gen_enso-Linear Regression: MAE=167.01
OK v7_only_gen_enso_lags-Linear Regression: MAE=167.01
OK v8_only_gen_no_solar_enso-Linear Regression: MAE=167.01
OK v8_only_gen_no_solar_enso_lags-XGBoost: MAE=64.07
OK v9_date_range_all-Decision Tree: MAE=27.88
OK v9_date_range_all_lags-Lasso: MAE=158.29
OK v10_date_range_no_solar-Decision Tree: MAE=112.21
OK v10_date_range_no_solar_lags-Lasso: MAE=158.29
